# v2 Persona Vector Pipeline — Extraction (Colab / Kaggle)

Runs the GPU-heavy extraction pass once per model: sub-test A + sub-test B prompt activations (all layers, both pooling variants, both formatting variants where available), plus Method 1's generation activations for instruct models. Resume-safe throughout — if this session gets evicted, just re-run from the top; already-completed work is never recomputed (see `src/checkpoint.py`).

**Prerequisites before this notebook can run:**
1. `Pipeline_v2/` and the `data/` CSVs (including `tone_pole_*.csv`, `subtest_b_*.csv`) must be pushed to the repo's remote — this notebook clones the repo fresh. Update `REPO_URL` below if the remote differs from `github.com/Fjord-H/Persona-Vector-Study`.
2. An `HF_TOKEN` secret (Colab: *Secrets* panel; Kaggle: *Add-ons → Secrets*) with license acceptance on file for `meta-llama/Llama-3.2-3B` and `meta-llama/Llama-3.2-3B-Instruct` — both are gated.
3. Nothing else — the unit tests (including the required unbatched-equivalence test) run automatically below and must pass before extraction starts.

In [ ]:
import os, sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None

REPO_URL = "https://github.com/Fjord-H/Persona-Vector-Study.git"  # update if the remote differs
REPO_DIR = pathlib.Path("/content/Persona-Vector-Study") if IN_COLAB else pathlib.Path("/kaggle/working/Persona-Vector-Study")

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

PIPELINE_DIR = REPO_DIR / "Pipeline_v2"
sys.path.insert(0, str(PIPELINE_DIR))
os.chdir(PIPELINE_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("environment:", "Colab" if IN_COLAB else ("Kaggle" if IN_KAGGLE else "unknown/local"))
print("working dir:", pathlib.Path.cwd())

In [ ]:
# Cache directory: Colab -> Google Drive (survives eviction), Kaggle -> /kaggle/working
# (persists for the session and can be saved as Kaggle output), local -> Pipeline_v2/cache.
# Overridable at any time by setting PV2_CACHE_DIR yourself before importing src.config.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    cache_root = pathlib.Path("/content/drive/MyDrive/persona_vector_v2_cache")
elif IN_KAGGLE:
    cache_root = pathlib.Path("/kaggle/working/pv2_cache")
else:
    cache_root = PIPELINE_DIR / "cache"

cache_root.mkdir(parents=True, exist_ok=True)
os.environ["PV2_CACHE_DIR"] = str(cache_root)
os.environ["PV2_MANIFEST_DIR"] = str(cache_root.parent / "pv2_manifests")
print("cache dir:", cache_root)

In [ ]:
# HF token — required for the gated Llama-3.2 models (base + instruct).
if IN_COLAB:
    from google.colab import userdata
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception as e:
        print("Could not read HF_TOKEN from Colab secrets:", e)
elif IN_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    try:
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print("Could not read HF_TOKEN from Kaggle secrets:", e)

assert os.environ.get("HF_TOKEN"), (
    "HF_TOKEN not set. Add it as a Colab/Kaggle secret named HF_TOKEN -- required for "
    "meta-llama/Llama-3.2-3B and -Instruct, both gated."
)
print("HF_TOKEN is set (", len(os.environ["HF_TOKEN"]), "chars )")

## Required: unit tests must pass before any real extraction

Per the spec's compute-constraints section — the unbatched-equivalence test in particular (`tests/test_pooling_batch_equivalence.py`) is the single highest-probability-of-a-new-bug spot given the length-channel finding in this project's data. Do not skip this cell.

In [ ]:
result = subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q"], cwd=PIPELINE_DIR)
assert result.returncode == 0, "Tests failed -- DO NOT proceed to real extraction until every test passes."

In [ ]:
# Frozen split: computed once (seeded, stratified, near-duplicate-grouped) and reused
# from here on. Safe to re-run -- it's a no-op once splits/splits_frozen.csv exists.
from src import splits
from src.config import FROZEN_SPLITS_PATH

splits_df = splits.load_frozen_splits()
print("frozen splits ready at", FROZEN_SPLITS_PATH)
print(splits_df.groupby(["label", "split"]).size().unstack(fill_value=0))

## Extraction

Edit `MODELS_TO_RUN` to extract a subset (useful for a first smoke run — start with `["gpt2-medium"]`, the smallest and only non-gated model, before spending quota on Qwen/Llama). Each model is fully resume-safe: re-running this cell after an eviction picks up exactly where it left off, per model and per formatting variant, with zero recomputation of already-cached items.

In [ ]:
from src.config import MODEL_REGISTRY
from src.run_extraction import extract_all_for_model

MODELS_TO_RUN = list(MODEL_REGISTRY.keys())
# MODELS_TO_RUN = ["gpt2-medium"]  # uncomment for a first smoke run

def progress(total, already_done, remaining, **_):
    print(f"    {already_done}/{total} done, {remaining} remaining", end="\r")

for model_key in MODELS_TO_RUN:
    print(f"=== {model_key} ===")
    manifest_paths = extract_all_for_model(model_key, progress_callback=progress)
    print()
    for p in manifest_paths:
        print("  wrote manifest:", p)

## Done

Everything after this point (method comparison, bootstrap CIs) is pure numpy/sklearn on the cache written above — no further GPU time needed, and it can run anywhere (including locally, off this cache directory). See `notebooks/02_method_comparison.ipynb`.

If using Colab with a Drive-mounted cache, the cache already persists across sessions. If using Kaggle's `/kaggle/working`, remember to save this notebook's output/version so the cache directory is retained as Kaggle output before the session ends.